In [51]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import ResNetForImageClassification,MobileNetV2ForImageClassification
from PIL import Image
import os
from pathlib import Path
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.optim import lr_scheduler
import random

In [52]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-3

In [ ]:

class CubeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.classes = ['red', 'green', 'blue']
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        self.images = []
        self.labels = []
        
        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                for img_path in class_dir.glob("*.png"):
                    self.images.append(img_path)
                    self.labels.append(self.class_to_idx[class_name])
        
        # Mostrar distribuição por classe
        class_counts = {cls: 0 for cls in self.classes}
        for label in self.labels:
            for cls, idx in self.class_to_idx.items():
                if label == idx:
                    class_counts[cls] += 1
        
        print(f"Dataset: {len(self.images)} imagens")
        for cls, count in class_counts.items():
            print(f"  {cls}: {count} imagens")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

import cv2  # Add this to your imports at the top

def rgb_to_hsv_transform():
    """Convert RGB to HSV color space"""
    def rgb_to_hsv(image):
        # Convert PIL to numpy
        img_array = np.array(image)
        
        # Convert RGB to HSV
        hsv_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2HSV)
        
        # Convert back to PIL
        return Image.fromarray(hsv_img)
    
    return transforms.Lambda(rgb_to_hsv)

def get_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5), 
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ✅ RGB normalization
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
      
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ✅ RGB normalization
    ])
    
    return train_transform, val_transform

In [54]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_score, model, path):
        save_path = Path(path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        
        if self.best_score is None:
            self.best_score = val_score
            torch.save(model.state_dict(), path)
            print(f"Modelo salvo: {save_path.name}")
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.counter = 0
            torch.save(model.state_dict(), path)
            print(f"Novo melhor modelo salvo: {save_path.name}")

In [55]:
def setup_finetuning(model, freeze_backbone=True):
    if freeze_backbone:
        try:
            # Tentar freezar backbone do MobileNetV2
            for param in model.mobilenet_v2.parameters():
                param.requires_grad = False
            print("MobileNetV2 backbone congelado")
        except AttributeError:
            try:
                # Alternativa: tentar 'backbone'
                for param in model.backbone.parameters():
                    param.requires_grad = False
                print("Backbone genérico congelado")
            except AttributeError:
                # Última tentativa: congelar tudo exceto classifier
                for name, param in model.named_parameters():
                    if 'classifier' not in name:
                        param.requires_grad = False
                print("Todas as camadas exceto classifier congeladas")
        
        # Só treina o classificador final
        for param in model.classifier.parameters():
            param.requires_grad = True
            
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Parâmetros treináveis: {trainable_params:,}")
    else:
        print("Fine-tuning completo ativado")
        
    return model

In [56]:
def train_fold(fold_idx, train_dataset, val_dataset, save_path):
    print(f"\nTreinando Fold {fold_idx + 1}")
    
    # GARANTIR que pasta existe
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                             num_workers=0, pin_memory=True)  # num_workers=0 para Windows
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=0, pin_memory=True)
    
    print(f"Batches: Train={len(train_loader)}, Val={len(val_loader)}")
    
    # Carregar modelo SEM .to(DEVICE) aqui
    model = MobileNetV2ForImageClassification.from_pretrained(
        "google/mobilenet_v2_1.0_224",
        num_labels=3,
        ignore_mismatched_sizes=True
    )
    
    model.config.id2label = {0: 'red', 1: 'green', 2: 'blue'}
    model.config.label2id = {'red': 0, 'green': 1, 'blue': 2}
    
    # Setup fine-tuning ANTES de mover para GPU
    model = setup_finetuning(model, freeze_backbone=True)
    
    # Mover para GPU só UMA vez
    model.to(DEVICE)
    
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=0.01)
    
    criterion = nn.CrossEntropyLoss()
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    early_stopping = EarlyStopping(patience=7)
    
    # Métricas de acompanhamento
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    for epoch in range(EPOCHS):
        # TREINO
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [TRAIN]")
        for images, labels in train_pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images).logits
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.4f}', 
                'Acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        train_acc = 100 * train_correct / train_total
        train_losses.append(train_loss / len(train_loader))
        train_accs.append(train_acc)
        
        # VALIDAÇÃO
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [VAL]")
            for images, labels in val_pbar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images).logits
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())
                
                val_pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'Acc': f'{100.*val_correct/val_total:.2f}%'
                })
        
        val_acc = 100 * val_correct / val_total
        val_f1 = f1_score(all_val_labels, all_val_preds, average='weighted')
        
        val_losses.append(val_loss / len(val_loader))
        val_accs.append(val_acc)
        
        scheduler.step()
        
        print(f' Epoch [{epoch+1}/{EPOCHS}] | '
              f'Train: {train_acc:.1f}% | Val: {val_acc:.1f}% | F1: {val_f1:.3f} | '
              f'LR: {scheduler.get_last_lr()[0]:.6f}')
        
        early_stopping(val_acc, model, save_path)
        if early_stopping.early_stop:
            print(f"Early stopping na época {epoch+1}")
            break
        
        # Limpar cache GPU a cada época
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Carregar melhor modelo
    if save_path.exists():
        model.load_state_dict(torch.load(save_path))
        print(f" Melhor modelo carregado: {early_stopping.best_score:.2f}%")
    else:
        print(f"Arquivo do modelo não encontrado: {save_path}")
    
    return {
        'fold': fold_idx,
        'best_val_acc': early_stopping.best_score,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'final_f1': val_f1
    }

In [57]:
def plot_fold_curves(fold_results, save_path=None):
    
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        epochs = range(len(fold_results['train_losses']))
        print(f"Epochs para plotar: {len(epochs)}")
        
        # Loss
        ax1.plot(epochs, fold_results['train_losses'], label='Train Loss', marker='o')
        ax1.plot(epochs, fold_results['val_losses'], label='Val Loss', marker='s')
        ax1.set_title(f'Loss Curves - Fold {fold_results["fold"] + 1}', fontsize=14)
        ax1.set_xlabel('Epochs')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Accuracy
        ax2.plot(epochs, fold_results['train_accs'], label='Train Acc', marker='o')
        ax2.plot(epochs, fold_results['val_accs'], label='Val Acc', marker='s')
        ax2.set_title(f'Accuracy Curves - Fold {fold_results["fold"] + 1}', fontsize=14)
        ax2.set_xlabel('Epochs')
        ax2.set_ylabel('Accuracy (%)')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Adicionar informações do fold
        val_acc = fold_results['best_val_acc']
        final_f1 = fold_results['final_f1']
        fig.suptitle(f'Fold {fold_results["fold"] + 1} - Best Val Acc: {val_acc:.2f}% | F1: {final_f1:.4f}', 
                     fontsize=16, fontweight='bold')
        
        plt.tight_layout()
        
        if save_path:
            # FORÇAR criação do diretório
            save_path = Path(save_path)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            
            try:
                plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
            except Exception as e1:
                try:
                    plt.savefig(str(save_path))
                except Exception as e2:
                    pass
           
            
        
        
        plt.close()  # Fecha figura
        print(f"Plot concluído para Fold {fold_results['fold'] + 1}")
        plt.close()
        
    except Exception as e:
        print(f"ERRO CRÍTICO ao criar plot: {e}")
        import traceback

In [58]:
# Função bonus para plot summary
def create_summary_plot(all_fold_results, save_path):
    """Cria um plot comparativo de todos os folds"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    colors = ['blue', 'red', 'green', 'orange', 'purple']
    
    # Plot 1: Loss comparison
    for i, fold_result in enumerate(all_fold_results):
        epochs = range(len(fold_result['train_losses']))
        ax1.plot(epochs, fold_result['val_losses'], label=f'Fold {i+1}', 
                color=colors[i % len(colors)], marker='o', markersize=3)
    ax1.set_title('Validation Loss - All Folds', fontsize=14)
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Accuracy comparison  
    for i, fold_result in enumerate(all_fold_results):
        epochs = range(len(fold_result['train_accs']))
        ax2.plot(epochs, fold_result['val_accs'], label=f'Fold {i+1}',
                color=colors[i % len(colors)], marker='s', markersize=3)
    ax2.set_title('Validation Accuracy - All Folds', fontsize=14)
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Best accuracies bar chart
    fold_names = [f'Fold {i+1}' for i in range(len(all_fold_results))]
    best_accs = [fold['best_val_acc'] for fold in all_fold_results]
    ax3.bar(fold_names, best_accs, color=colors[:len(all_fold_results)])
    ax3.set_title('Best Validation Accuracy per Fold', fontsize=14)
    ax3.set_ylabel('Accuracy (%)')
    for i, acc in enumerate(best_accs):
        ax3.text(i, acc + 0.5, f'{acc:.2f}%', ha='center', va='bottom')
    
    # Plot 4: F1 scores
    f1_scores = [fold['final_f1'] for fold in all_fold_results]
    ax4.bar(fold_names, f1_scores, color=colors[:len(all_fold_results)])
    ax4.set_title('Final F1 Score per Fold', fontsize=14)
    ax4.set_ylabel('F1 Score')
    for i, f1 in enumerate(f1_scores):
        ax4.text(i, f1 + 0.01, f'{f1:.4f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    


In [59]:
def run_kfold():
    print(f"Dispositivo: {DEVICE}")
    print(f"Configurações: Batch={BATCH_SIZE}, Epochs={EPOCHS}, LR={LEARNING_RATE}")
    
    folds_dir = Path(r"C:\Users\User\Documents\IA_PROJECT\Archives\files_webot\IA_20252\controllers\Model_Processing\cubes_classification_folds")
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)
    
    # Criar pasta para plots
    plots_dir = results_dir / "plots"
    plots_dir.mkdir(exist_ok=True)
    
    train_transform, val_transform = get_transforms()
    fold_results = []
    fold_accuracies = []
    
    num_folds = 2
    
    for fold_idx in range(num_folds):
        fold_path = folds_dir / f"fold_{fold_idx}"
        
        train_dataset = CubeDataset(fold_path / "train", transform=train_transform)
        val_dataset = CubeDataset(fold_path / "val", transform=val_transform)
        
        model_save_path = results_dir / f"best_model_fold_{fold_idx}.pth"
        fold_result = train_fold(fold_idx, train_dataset, val_dataset, model_save_path)
        
        fold_results.append(fold_result)
        fold_accuracies.append(fold_result['best_val_acc'])
        
        # Salvar plot como arquivo
        plot_save_path = plots_dir / f"fold_{fold_idx}_curves.png"
        plot_fold_curves(fold_result, save_path=plot_save_path)
        
        # Limpar cache da GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Resultados finais
    print(f"\n{'='*60}")
    print(f"RESULTADOS {num_folds}-FOLD CROSS-VALIDATION")
    print(f"{'='*60}")
    
    for i, result in enumerate(fold_results):
        print(f"Fold {i+1}: {result['best_val_acc']:.2f}% (F1: {result['final_f1']:.4f})")
    
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    
    print(f"\nAcurácia Média: {mean_acc:.2f} ± {std_acc:.2f}%")
    print(f"Melhor Fold: {max(fold_accuracies):.2f}%")
    print(f"Pior Fold: {min(fold_accuracies):.2f}%")
    
    best_fold_idx = np.argmax(fold_accuracies)
    print(f"Melhor Modelo: Fold {best_fold_idx + 1}")
    
    # Criar plot summary de todos os folds
    create_summary_plot(fold_results, plots_dir / "summary_all_folds.png")
    
    # ===== TESTE FINAL =====
    # Verificar se existe pasta de teste
    test_data_path = folds_dir / "test"  # ou onde estiver seu conjunto de teste
    
    # Se não existir pasta test nos folds, tentar usar um fold como teste final
    if not test_data_path.exists():
        print(f"Pasta de teste não encontrada em: {test_data_path}")
        print(f"Usando Fold {best_fold_idx + 1} como referência de melhor desempenho")
        final_test_results = None
    else:
        best_model_file = results_dir / f"best_model_fold_{best_fold_idx}.pth"
        final_test_results = test_final_model(best_model_file, test_data_path, results_dir)
    
    # Salvar resultados
    final_results = {
        'mean_accuracy': float(mean_acc),
        'std_accuracy': float(std_acc),
        'fold_accuracies': [float(acc) for acc in fold_accuracies],
        'best_fold': int(best_fold_idx),
        'best_model_path': str(results_dir / f"best_model_fold_{best_fold_idx}.pth"),
        'plots_directory': str(plots_dir),
        'all_fold_results': fold_results,
        'final_test_results': final_test_results
    }
    
    with open(results_dir / "results.json", "w") as f:
        json.dump(final_results, f, indent=2, default=str)
    
    # Salvar resultados do teste final separadamente se existir
    if final_test_results:
        with open(results_dir / "final_test_results.json", "w") as f:
            json.dump(final_test_results, f, indent=2, default=str)
        print(f"Resultados do teste final salvos!")
    
    print(f"\nArquivos salvos em: {results_dir}")
    print(f"Plots salvos em: {plots_dir}")
    
    return final_results

In [60]:

def test_single_image(model_path, image_path):
    model = MobileNetV2ForImageClassification.from_pretrained(
        "google/mobilenet_v2_1.0_224", 
        num_labels=3,
        ignore_mismatched_sizes=True
    )
    model.load_state_dict(torch.load(model_path))
    model.to(DEVICE)
    model.eval()
    
    _, val_transform = get_transforms()
    image = Image.open(image_path).convert('RGB')
    image_tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(image_tensor).logits
        predicted = torch.max(outputs, 1)[1].item()
        probabilities = torch.softmax(outputs, dim=1)[0]
    
    classes = ['red', 'green', 'blue']
    predicted_class = classes[predicted]
    confidence = probabilities[predicted].item() * 100
    
    print(f"Predição: {predicted_class} ({confidence:.2f}%)")
    
    for i, cls in enumerate(classes):
        prob = probabilities[i].item() * 100
        print(f"  {cls}: {prob:.2f}%")



In [61]:
def test_final_model(best_model_path, test_data_path, save_results_dir=None):
    """
    Testa o melhor modelo em um conjunto de teste final
    """
    # Carregar modelo
    model = MobileNetV2ForImageClassification.from_pretrained(
        "google/mobilenet_v2_1.0_224",
        num_labels=3,
        ignore_mismatched_sizes=True
    )
    
    # Configurar labels como no treinamento
    model.config.id2label = {0: 'red', 1: 'green', 2: 'blue'}
    model.config.label2id = {'red': 0, 'green': 1, 'blue': 2}
    
    # Setup fine-tuning para manter compatibilidade
    model = setup_finetuning(model, freeze_backbone=True)
    
    if Path(best_model_path).exists():
        model.load_state_dict(torch.load(best_model_path))
    else:
        return None
    
    model.to(DEVICE)
    model.eval()
    
    # Carregar dados de teste
    _, test_transform = get_transforms()
    test_dataset = CubeDataset(test_data_path, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Teste
    all_preds = []
    all_labels = []
    test_correct = 0
    test_total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images).logits
            _, predicted = torch.max(outputs.data, 1)
            
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcular métricas finais
    test_accuracy = 100 * test_correct / test_total
    test_f1 = f1_score(all_labels, all_preds, average='weighted')
    
    # Relatório detalhado
    class_names = ['red', 'green', 'blue']
    report = classification_report(all_labels, all_preds, 
                                 target_names=class_names, 
                                 output_dict=True)
    
    # Matriz de confusão
    cm = confusion_matrix(all_labels, all_preds)
    
    # Salvar resultados do teste final
    final_test_results = {
        'test_accuracy': float(test_accuracy),
        'test_f1_weighted': float(test_f1),
        'model_used': str(best_model_path),
        'test_data_path': str(test_data_path),
        'classification_report': report,
        'confusion_matrix': cm.tolist(),
        'total_test_samples': int(test_total)
    }
    
    # Salvar em arquivo específico se diretório fornecido
    if save_results_dir:
        save_path = Path(save_results_dir)
        save_path.mkdir(parents=True, exist_ok=True)
        
        # Extrair fold number do model path
        fold_num = Path(best_model_path).stem.split('_')[-1]
        test_results_file = save_path / f"test_results_fold_{fold_num}.json"
        
        with open(test_results_file, "w") as f:
            json.dump(final_test_results, f, indent=2, default=str)
    
    return final_test_results

In [ ]:
# EXECUTAR O PIPELINE COMPLETO
if __name__ == "__main__":
    
    results = run_kfold()
    
 
    
    print(f"\nPipeline completo finalizado!")
    print(f"Acurácia média do K-Fold: {results['mean_accuracy']:.2f}%")
    
    if results['final_test_results']:
        test_acc = results['final_test_results']['test_accuracy']
        print(f"Acurácia no teste final: {test_acc:.2f}%")
    else:
        print(f"Melhor fold foi o {results['best_fold'] + 1}")
    
    print(f"\nTodos os resultados foram salvos na pasta 'results/'")

Dispositivo: cpu
Configurações: Batch=32, Epochs=5, LR=0.001
Dataset: 7050 imagens
  red: 2350 imagens
  green: 2350 imagens
  blue: 2350 imagens
Dataset: 1764 imagens
  red: 588 imagens
  green: 588 imagens
  blue: 588 imagens

Treinando Fold 1
Batches: Train=221, Val=56


Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([3, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MobileNetV2 backbone congelado
Parâmetros treináveis: 3,843


Epoch 1/5 [TRAIN]:   0%|          | 0/221 [00:00<?, ?it/s]c:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 1/5 [VAL]: 100%|██████████| 56/56 [00:19<00:00,  2.94it/s, Loss=0.0567, Acc=99.38%] 


 Epoch [1/5] | Train: 95.5% | Val: 99.4% | F1: 0.994 | LR: 0.000905
Modelo salvo: best_model_fold_0.pth


...

Epoch 1/5 [TRAIN]:   0%|          | 0/221 [00:00<?, ?it/s]c:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 1/5 [VAL]: 100%|██████████| 56/56 [00:24<00:00,  2.27it/s, Loss=0.6643, Acc=97.34%]


 Epoch [1/5] | Train: 94.9% | Val: 97.3% | F1: 0.973 | LR: 0.000905
Modelo salvo: best_model_fold_1.pth


Epoch 2/5 [TRAIN]:   0%|          | 0/221 [00:00<?, ?it/s]c:\Users\User\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 2/5 [TRAIN]:  74%|███████▍  | 164/221 [01:02<00:21,  2.62it/s, Loss=0.0088, Acc=98.69%]


KeyboardInterrupt: 

: 